# Raccolta Dati da Reddit — r/Italia, keyword: *film*
Raccolta tramite **Arctic Shift** (archivio pubblico Reddit per ricerca accademica).
Nessuna credenziale richiesta.

**Strategia**: cerchiamo direttamente i **commenti** che contengono la parola 'film'
in r/Italia, limitando a 300 commenti per anno (2023, 2022, 2021, 2020, 2019) fino a raggiungere un totale di 700 commenti.

## Installazione dipendenze

In [2]:
# !pip install requests spacy tqdm
# !python -m spacy download it_core_news_sm

## Configurazione

In [2]:
import requests
import time
import pandas as pd
from datetime import datetime

SUBREDDIT       = "Italia"
KEYWORD         = "film"
TARGET_COMMENTS = 700
OUTPUT_CSV      = f"corpus_{SUBREDDIT}_{KEYWORD}.csv"

BASE_URL = "https://arctic-shift.photon-reddit.com/api"
HEADERS  = {"User-Agent": "python:elnsm.progetto.film:v1.0 (academic NLP project)"}

print("Configurazione pronta.")
print(f"  Subreddit : r/{SUBREDDIT}")
print(f"  Keyword   : '{KEYWORD}'")
print(f"  Obiettivo : {TARGET_COMMENTS} commenti")

Configurazione pronta.
  Subreddit : r/Italia
  Keyword   : 'film'
  Obiettivo : 700 commenti


## Test connessione API
Attenzione, con servizi intermedi come **Arctic Shift** le risposte possono variare. Se lo status è 200 OK, altrimenti se è 422 riprova dopo qualche secondo.

In [6]:
# Verifica che l'API risponda correttamente prima di procedere
test_url = f"{BASE_URL}/comments/search"
test_params = {
    "subreddit": SUBREDDIT,
    "body"   : KEYWORD,
    "limit"    : 3,
    "after"    : "2022-01-01",
    "before"   : "2022-12-31"
}

resp = requests.get(test_url, headers=HEADERS, params=test_params, timeout=20)
print(f"Status: {resp.status_code}")

if resp.status_code == 200:
    data = resp.json()
    print(f"Risposta OK! Esempio commento:")
    if data.get("data"):
        print(data["data"][0].get("body", "")[:200])
    print(f"\nChiavi disponibili: {list(data['data'][0].keys()) if data.get('data') else 'nessuna'}")
else:
    print(f"Errore: {resp.text}")

Status: 200
Risposta OK! Esempio commento:
L’ha letteralmente fatto ieri/l’altro ieri, vuoi lo screen? Poi sarei io quella che si crea i film mentali per avere ragione. Ci sono migliaia di cose disgustose che ha fatto e detto di cui ti possono

Chiavi disponibili: ['all_awardings', 'archived', 'associated_award', 'author', 'author_created_utc', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_richtext', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'author_flair_type', 'author_fullname', 'author_patreon_flair', 'author_premium', 'body', 'can_gild', 'collapsed', 'collapsed_because_crowd_control', 'collapsed_reason', 'collapsed_reason_code', 'comment_type', 'controversiality', 'created_utc', 'distinguished', 'edited', 'gilded', 'gildings', 'id', 'is_submitter', 'link_id', 'locked', 'name', 'no_follow', 'parent_id', 'permalink', 'retrieved_on', 'score', 'score_hidden', 'send_replies', 'stickied', 'subreddit', 'subreddit_id', 'subreddit

## Raccolta commenti
Questa cella è la più critica, la raccolta di commenti tramite API può essere lenta e soggetta a errori di rete o limiti di rate. Anche qui il risultato puo variare, ripetere quindi finche non si è soddisfatti.

Questo codice implementa una raccolta per finestre temporali annuali, con un limite di 300 commenti per anno, fino a raggiungere un totale di 700 commenti. Se il limite annuale viene raggiunto, si passa alla finestra successiva. Se il target totale viene raggiunto, si interrompe la raccolta.

In [7]:
def collect_comments(subreddit, keyword, target=700):
    """
    Raccoglie commenti da Arctic Shift con un limite di 250 per anno.
    """
    all_comments = []
    LIMIT_PER_YEAR = 300  # <-- Limite richiesto

    time_windows = [
        ("2023-01-01", "2024-01-01"),
        ("2022-01-01", "2023-01-01"),
        ("2021-01-01", "2022-01-01"),
        ("2020-01-01", "2021-01-01"),
        ("2019-01-01", "2020-01-01"),
    ]

    for after, before in time_windows:
        # Se abbiamo già raggiunto il target totale generale, possiamo fermarci del tutto
        if len(all_comments) >= target:
            break

        print(f"\nFinestra {after[:4]}: inizio raccolta (max {LIMIT_PER_YEAR})...")

        params = {
            "subreddit": subreddit,
            "body": keyword,
            "limit": 100,
            "after": after,
            "before": before,
            "sort": "desc"
        }

        window_comments = []
        last_utc = None

        while len(window_comments) < LIMIT_PER_YEAR: # <-- Controllo limite annuale
            if last_utc:
                params["before"] = last_utc

            try:
                resp = requests.get(
                    f"{BASE_URL}/comments/search",
                    headers=HEADERS,
                    params=params,
                    timeout=20
                )
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                print(f"  Errore: {e}")
                break

            items = data.get("data", [])
            if not items:
                break

            for c in items:
                # Controlliamo il limite anche dentro il loop per non superare i 250
                if len(window_comments) >= LIMIT_PER_YEAR:
                    break

                body = c.get("body", "").strip()
                if body in ("", "[deleted]", "[removed]") or len(body) < 15:
                    continue

                ts = float(c.get("created_utc", 0))
                window_comments.append({
                    "post_id"          : c.get("link_id", "").replace("t3_", ""),
                    "post_title"       : c.get("link_title", ""),
                    "comment_id"       : c.get("id", ""),
                    "comment_text"     : body,
                    "comment_author"   : c.get("author", ""),
                    "comment_timestamp": datetime.utcfromtimestamp(ts).isoformat(),
                    "comment_score"    : c.get("score", 0),
                    "subreddit"        : c.get("subreddit", subreddit),
                    "permalink"        : "https://reddit.com" + c.get("permalink", "")
                                         if c.get("permalink") else ""
                })

            # Prepara il cursore per la pagina successiva
            oldest_utc = min(float(c.get("created_utc", 0)) for c in items)
            last_utc = datetime.utcfromtimestamp(oldest_utc).strftime("%Y-%m-%dT%H:%M:%S")

            print(f"  Scaricati {len(window_comments)} commenti per il {after[:4]}...", end="\r")

            if len(items) < 100:
                break

            time.sleep(0.8)

        # Aggiungiamo i commenti dell'anno alla lista generale
        all_comments.extend(window_comments)
        print(f"  -> Concluso {after[:4]}: {len(window_comments)} commenti. Totale generale: {len(all_comments)}")
        time.sleep(1)

    return all_comments

print("Funzione definita con limite di 300 per anno. Avvio...")
raw_comments = collect_comments(SUBREDDIT, KEYWORD, target=TARGET_COMMENTS)

Funzione definita con limite di 300 per anno. Avvio...

Finestra 2023: inizio raccolta (max 300)...
  -> Concluso 2023: 300 commenti. Totale generale: 300

Finestra 2022: inizio raccolta (max 300)...
  -> Concluso 2022: 300 commenti. Totale generale: 600

Finestra 2021: inizio raccolta (max 300)...
  -> Concluso 2021: 146 commenti. Totale generale: 746


## Salvataggio CSV
Salviamo i commenti raccolti in un file CSV.

In [11]:
# Converti la lista di dict raccolta dall'API in DataFrame
df_raw = pd.DataFrame(raw_comments)

if df_raw.empty:
    print("Nessun commento raccolto: controlla parametri API o riprova la raccolta.")
else:
    df_corpus = df_raw.drop_duplicates(subset="comment_id").reset_index(drop=True)
    df_corpus.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"Salvati {len(df_corpus)} commenti unici in '{OUTPUT_CSV}'")
    print(f"Colonne: {list(df_corpus.columns)}")
    display(df_corpus[["post_title", "comment_text", "comment_author", "comment_timestamp"]].head(5))

Salvati 746 commenti unici in 'corpus_Italia_film.csv'
Colonne: ['post_id', 'post_title', 'comment_id', 'comment_text', 'comment_author', 'comment_timestamp', 'comment_score', 'subreddit', 'permalink']


,post_title,comment_text,comment_author,comment_timestamp
0,,Ho visto il film Hackers e da informatico l'ho...,centomila,2023-12-31T23:48:28
1,,Non l'ho visto ma mi immagino abbia detto semp...,Vivotrapazzi,2023-12-31T23:38:13
2,,"Se lo leggi in italiano, le ultime opere sono ...",Vivotrapazzi,2023-12-31T23:35:17
3,,Film e birra. Buon anno!,sal_gub,2023-12-31T23:14:01
4,,Si poteva non esserci il film carino o sarei p...,Altarus12,2023-12-31T22:21:30


## Tokenizzazione, Lemmatizzazione e POS-tagging
Utilizziamo il modello italiano `it_core_news_sm` di spaCy per analizzare i commenti. Per ogni token estraiamo:
- testo originale (`token.text`)
- lemma in minuscolo (`token.lemma_.lower()`)
- parte del discorso (`token.pos_`)
- flag booleano se è un aggettivo (`token.pos_ == "ADJ"`)
- filtriamo spazi e punteggiatura

Eseguiamo un test su un commento di esempio per verificare l'output e infine processiamo tutti i commenti e salviamo i token in un CSV separato. Mostriamo poi le statistiche sugli aggettivi più frequenti.

Salviamo i risultati in un DataFrame per ulteriori analisi.


In [12]:
import spacy

nlp = spacy.load("it_core_news_sm")

def process_text(text):
    doc = nlp(str(text))
    return [
        {
            "token" : token.text,
            "lemma" : token.lemma_.lower(),
            "pos"   : token.pos_,
            "is_adj": token.pos_ == "ADJ"
        }
        for token in doc
        if not token.is_space and not token.is_punct
    ]

# Test su un commento di esempio
esempio = df_corpus["comment_text"].iloc[0]
print(f"Commento:\n{esempio}\n")
print("Analisi:")
for t in process_text(esempio):
    flag = " <- ADJ" if t["is_adj"] else ""
    print(f"  {t['token']:20s} | {t['lemma']:20s} | {t['pos']}{flag}")

Commento:
Ho visto il film Hackers e da informatico l'ho trovato discutibile in alcuni messaggi.

(C'è infinito potenziale qui)

Analisi:
  Ho                   | avere                | AUX
  visto                | vedere               | VERB
  il                   | il                   | DET
  film                 | film                 | NOUN
  Hackers              | hackers              | PROPN
  e                    | e                    | CCONJ
  da                   | da                   | ADP
  informatico          | informatico          | NOUN
  l'                   | lo                   | PRON
  ho                   | avere                | AUX
  trovato              | trovare              | VERB
  discutibile          | discutibile          | ADJ <- ADJ
  in                   | in                   | ADP
  alcuni               | alcuno               | DET
  messaggi             | messaggio            | NOUN
  C'                   | c'                   | PRON
  è         

In [13]:
from tqdm.auto import tqdm

print(f"Processamento di {len(df_corpus)} commenti con spaCy...")

all_tokens = []
for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus)):
    for t in process_text(row["comment_text"]):
        all_tokens.append({"comment_id": row["comment_id"], **t})

df_tokens = pd.DataFrame(all_tokens)
df_adj    = df_tokens[df_tokens["is_adj"] == True]

print(f"\nToken totali     : {len(df_tokens)}")
print(f"Aggettivi trovati: {len(df_adj)}")
print(f"\nTop 20 aggettivi più frequenti:")
print(df_adj["lemma"].value_counts().head(20))

/Users/veronicabosso/PycharmProjects/ELIta_tesi/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processamento di 746 commenti con spaCy...


100%|██████████| 746/746 [00:11<00:00, 65.07it/s] 


Token totali     : 62282
Aggettivi trovati: 4172

Top 20 aggettivi più frequenti:
lemma
altro           101
stesso          100
bello            79
primo            78
italiano         57
nuovo            55
migliore         49
vero             48
ultimo           40
americano        39
originale        38
grande           37
buono            35
diverso          33
giusto           32
unico            32
vecchio          30
simile           27
interessante     26
secondo          26
Name: count, dtype: int64


In [14]:
tokens_file = f"tokens_{SUBREDDIT}_{KEYWORD}.csv"
df_tokens.to_csv(tokens_file, index=False, encoding="utf-8-sig")
print(f"Token salvati in '{tokens_file}'")

Token salvati in 'tokens_Italia_film.csv'


## Statistiche del corpus

In [15]:
import plotly.express as px

print("=" * 50)
print("STATISTICHE CORPUS")
print("=" * 50)
print(f"Commenti totali          : {len(df_corpus)}")
print(f"Post unici               : {df_corpus['post_id'].nunique()}")
print(f"Autori unici             : {df_corpus['comment_author'].nunique()}")
print(f"Token totali             : {len(df_tokens)}")
print(f"Aggettivi totali         : {len(df_adj)}")
print(f"Aggettivi unici (lemmi)  : {df_adj['lemma'].nunique()}")
lunghezze = df_corpus["comment_text"].str.split().str.len()
print(f"Lunghezza media commenti : {lunghezze.mean():.1f} parole")
print(f"Periodo temporale        : {df_corpus['comment_timestamp'].min()[:10]} → {df_corpus['comment_timestamp'].max()[:10]}")

# Distribuzione per anno
df_corpus["anno"] = df_corpus["comment_timestamp"].str[:4]
fig1 = px.bar(
    df_corpus["anno"].value_counts().sort_index().reset_index(),
    x="anno", y="count",
    title="Commenti per anno",
    labels={"anno": "Anno", "count": "Numero commenti"}
)
fig1.show()

# Distribuzione lunghezza commenti
fig2 = px.histogram(
    x=lunghezze, nbins=40,
    title="Distribuzione lunghezza commenti (parole)",
    labels={"x": "Parole", "y": "Commenti"}
)
fig2.show()

STATISTICHE CORPUS
Commenti totali          : 746
Post unici               : 323
Autori unici             : 517
Token totali             : 62282
Aggettivi totali         : 4172
Aggettivi unici (lemmi)  : 1563
Lunghezza media commenti : 81.9 parole
Periodo temporale        : 2021-01-01 → 2023-12-31
